Do all analysis in this notebook file
1. Read data

In [1]:
import os
import pandas as pd
from online_retail.utils.base_funcs import load_data

path = os.getcwd()
file_path = os.path.abspath(
    os.path.join(path, '..','..','data/online_retail_II.xlsx'))
data = load_data(file_path=file_path)
data.rename(columns = {'Customer ID':'CustomerID'}, inplace = True)
data.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Retail data has 8 features. Info about the data is also provided in following parts:

In [2]:
print(data.columns)
data.head()

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country'],
      dtype='object')


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
data.info()
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   CustomerID   417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
CustomerID     107927
Country             0
dtype: int64

Some stocks does not have description. Some customers bought as a guest so they do not have customer ID. Use SQL to write queries. At first, I want to see whether the Price column is the unit price or is Quantity * U-price. For that, I write a query for just one Stock, within a limited time, to see if we get different values or not. 
Some Quantities are negative which shows it is return objects or canceled (the invoice starts with C). For the same Stock number, the value is different. I guess it contains some charges like shipment. For those that the price is lower, we can say it may contians some discount. I guess we can consider the mode value for the true value of the Stock. At the end, I think it is the price column shows the unit price.

In [4]:
import pandasql as psql

query = """
SELECT *
FROM data
WHERE StockCode = '85048'
AND InvoiceDate BETWEEN '2009-12-01' AND '2010-02-01'
ORDER BY Country
"""

result = psql.sqldf(query, locals())
result.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,492746,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,4,2009-12-18 13:01:00.000000,7.95,NaN,EIRE
1,492761,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,8,2009-12-18 14:22:00.000000,7.95,14911.0,EIRE
2,C493859,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,-4,2010-01-07 16:15:00.000000,7.95,14911.0,EIRE
3,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00.000000,6.95,13085.0,United Kingdom
4,C489518,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,-1,2009-12-01 11:35:00.000000,7.95,15461.0,United Kingdom


Let us do RFM analysis.<br>
R -> Recency: How recently a customer made their last purchase -> more engaged and more likely to buy again.<br>
F -> Frequecy: How often a customer purchase -> Loyal customers.<br>
M -> Monetary: How much they spent -> spend a lot have more value.<br>
This is a study that is used for customer segmentation technique. The goal is to quantify customer value and behavior. So using these three criteria, we can segment our customers into 5 categories.<br>
1. Champions -> High RFM -> loyal and active.<br>
2. Loyal -> High F but regular M -> Regular customer.<br>
3. Big Spenders -> High M low F -> High value but occasional<br>
4. At Risk -> Used to buy but not now -> At risk for churn<br>
5. Lost -> Have not purchased for a long time.<br>

Let us write different querie to make the table for RFM.

In [ ]:
query = """
WITH temp AS(
SELECT InvoiceDate as snapshot_date
FROM data
WHERE InvoiceDate = max(InvoiceDate)
LIMIT 1
)

SELECT CustomerID, max(InvoiceDate) Last_purchase,
t.snapshot_date-max(InvoiceDate) Recency
FROM data
JOIN temp t
GROUP BY CustomerID
"""

R = psql.sqldf(query, locals())

PandaSQLException: (sqlite3.OperationalError) incomplete input
[SQL: 
WITH temp AS(
SELECT InvoiceDate as snapshot_date
FROM data
WHERE InvoiceDate = max(InvoiceDate)
LIMIT 1
)

]
(Background on this error at: https://sqlalche.me/e/20/e3q8)